In [ ]:
!pip install osmnx networkx deap torch numpy pandas matplotlib

In [ ]:
# ================================
# Pipeline EMS complet : NSGA-II hybride + RL
# ================================
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
import random, time
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from deap import base, creator, tools, algorithms
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ----------------------------
# 0. Réglages
# ----------------------------
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
sns.set(style="whitegrid")

# ----------------------------
# 1. RAW EMS DATASET
# ----------------------------
print("Loading EMS dataset...")
df = pd.read_csv("EMS_dataset.csv")  # adapter le chemin si besoin
print(f"Initial rows: {len(df)}")

# ----------------------------
# 2. DATA CLEANING & FEATURE ENGINEERING
# ----------------------------
# Nettoyage des coordonnées manquantes
df = df.dropna(subset=["Latitude", "Longitude"]).copy()

# Conversion datetime
df["INCIDENT_DATETIME"] = pd.to_datetime(df["INCIDENT_DATETIME"], errors="coerce")
df = df.dropna(subset=["INCIDENT_DATETIME"]).copy()

# Extraction des features temporelles
df["Hour"] = df["INCIDENT_DATETIME"].dt.hour
df["DayOfWeek"] = df["INCIDENT_DATETIME"].dt.dayofweek
df["Month"] = df["INCIDENT_DATETIME"].dt.month

# Encodage des types d’appels
call_types = df["INITIAL_CALL_TYPE"].fillna("UNKNOWN").unique()
call_map = {ct: i for i, ct in enumerate(call_types)}
df["Call_Code"] = df["INITIAL_CALL_TYPE"].map(call_map)

# Attribution des priorités (exemple simple, adapter selon expertise médicale)
priority_map = {"INJURY": 3, "MVAINJ": 2, "DRUG": 1}
df["Priority"] = df["INITIAL_CALL_TYPE"].map(priority_map).fillna(1).astype(int)

# ----------------------------
# 3. SPATIAL MAPPING (OSMnx)
# ----------------------------
place = "Manhattan, New York, USA"
print("Downloading street network (this can take time)...")
G = ox.graph_from_place(place, network_type="drive")
G = ox.add_edge_lengths(G)  # assure que 'length' existe

# Precompute node coordinates mapping for plotting
node_xy = {n: (data["x"], data["y"]) for n, data in G.nodes(data=True)}

# ----------------------------
# 4. MAP INCIDENTS TO NEAREST NODES
# ----------------------------
# To speed up, sample a subset for mapping if dataset is huge
sample_for_mapping = df.sample(min(len(df), 200000), random_state=42)  # adjust
print("Mapping incidents to nearest graph nodes (sample)...")
# ox.distance.nearest_nodes expects x (lon), y (lat)
incident_nodes_mapped = ox.distance.nearest_nodes(G,
                                                  X=sample_for_mapping["Longitude"].values,
                                                  Y=sample_for_mapping["Latitude"].values)
sample_for_mapping = sample_for_mapping.reset_index(drop=True)
sample_for_mapping["node"] = incident_nodes_mapped

# Aggregate true counts per node and hour for demand ground truth
agg_true = sample_for_mapping.groupby(["node", "Hour"]).size().reset_index(name="count")

# ----------------------------
# 5. DEMAND MODELING (Probabilistic)
# ----------------------------
# Simple probabilistic estimator: empirical frequency per (node,hour)
agg_true["prob"] = agg_true["count"] / agg_true["count"].sum()

# Build a dictionary for quick lookup: (node,hour) -> (count, prob)
demand_dict = {(row["node"], int(row["Hour"])): (int(row["count"]), float(row["prob"]))
               for _, row in agg_true.iterrows()}

# For global node-level demand (ignoring hour) for some metrics
node_counts = agg_true.groupby("node")["count"].sum().reset_index()
node_counts["prob_node"] = node_counts["count"] / node_counts["count"].sum()

In [ ]:
# ----------------------------
# 6. RL ENVIRONMENT
# ----------------------------
class EMSEnv:
    def __init__(self, G, ambulances, demand_dict, node_counts):
        self.G = G
        self.ambulances = ambulances  # list of node ids where ambulances are
        self.demand_dict = demand_dict
        self.node_counts = node_counts.set_index("node") if isinstance(node_counts, pd.DataFrame) else node_counts
        self.state_dim = len(ambulances)

    def reset(self):
        # state: normalized positions (node indices -> small vector)
        return np.random.rand(self.state_dim).astype(np.float32)

    def sample_incident(self):
        # sample a node proportional to node-level counts
        nodes = list(self.node_counts.index)
        probs = self.node_counts["prob_node"].values
        probs = probs / probs.sum()
        chosen = np.random.choice(nodes, p=probs)
        return chosen

    def compute_response_time_minutes(self, ambulance_node, incident_node, speed_kmh=40):
        try:
            length = nx.shortest_path_length(self.G, ambulance_node, incident_node, weight="length")
            time_min = (length / 1000) / speed_kmh * 60
            return time_min
        except Exception:
            return 1e6

    def step(self, action):
        ambulance_node = self.ambulances[action]
        incident_node = self.sample_incident()
        T = self.compute_response_time_minutes(ambulance_node, incident_node)
        # reward: negative response time, penalize long times; can incorporate demand weight
        demand_weight = self.node_counts.loc[incident_node, "prob_node"] if incident_node in self.node_counts.index else 0
        reward = -T * (1 + demand_weight)
        next_state = np.random.rand(self.state_dim).astype(np.float32)
        done = np.random.rand() > 0.98
        info = {"incident_node": incident_node, "response_time": T}
        return next_state, reward, done, info


In [ ]:
# ----------------------------
# 7. RL AGENT (DQN) & TRAINING (avec suivi loss)
# ----------------------------
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.net(x)

def train_rl(env, episodes=50, max_steps=50, epsilon_start=0.9, epsilon_end=0.05, epsilon_decay=0.995):
    device = torch.device("cpu")
    model = DQN(env.state_dim, env.state_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    rewards_history = []
    losses_history = []
    eps = epsilon_start

    for ep in range(episodes):
        state = torch.FloatTensor(env.reset()).to(device)
        total_reward = 0.0
        for step in range(max_steps):
            q_values = model(state)
            if np.random.rand() < eps:
                action = np.random.randint(0, env.state_dim)
            else:
                action = torch.argmax(q_values).item()

            next_state, reward, done, info = env.step(action)
            next_state_t = torch.FloatTensor(next_state).to(device)

            # target: r + gamma * max Q(next)
            with torch.no_grad():
                target_q = reward + 0.95 * torch.max(model(next_state_t)).item()
            pred_q = q_values[action]

            loss = loss_fn(pred_q, torch.tensor(target_q).to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses_history.append(loss.item())
            total_reward += reward
            state = next_state_t
            if done:
                break

        rewards_history.append(total_reward)
        eps = max(epsilon_end, eps * epsilon_decay)

    return model, rewards_history, losses_history


In [ ]:
# ----------------------------
# 8. NSGA-II OPTIMIZATION (placement global)
# ----------------------------
def evaluate_solution(individual, incidents_nodes, priorities_map, G, speed_kmh=40):
    # individual: list of ambulance node assignments (one ambulance per incident)
    f_time, f_weighted = 0.0, 0.0
    for j, inc_node in enumerate(incidents_nodes):
        amb_node = individual[j]
        try:
            length = nx.shortest_path_length(G, amb_node, inc_node, weight="length")
            T = (length / 1000) / speed_kmh * 60
        except Exception:
            T = 1e6
        p = priorities_map.get(inc_node, 1)
        f_time += T
        f_weighted += p * T
    return f_time, f_weighted

# DEAP setup
if "FitnessMulti" not in creator.__dict__:
    creator.create("FitnessMulti", base.Fitness, weights=(-1.0, -1.0))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMulti)

toolbox = base.Toolbox()
toolbox.register("attr_node", lambda nodes: random.choice(nodes))
# We'll register individual/population later per scenario

def run_nsga2(incidents_nodes, ambulance_nodes, priorities_map, G,
              pop_size=40, ngen=30, cxpb=0.7, mutpb=0.2):
    # register with current nodes
    toolbox.register("individual", tools.initRepeat, creator.Individual,
                     lambda: random.choice(ambulance_nodes), n=len(incidents_nodes))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    def eval_ind(ind):
        return evaluate_solution(ind, incidents_nodes, priorities_map, G)

    toolbox.register("evaluate", eval_ind)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05)
    toolbox.register("select", tools.selNSGA2)

    pop = toolbox.population(n=pop_size)
    algorithms.eaMuPlusLambda(pop, toolbox, mu=pop_size, lambda_=pop_size*2,
                              cxpb=cxpb, mutpb=mutpb, ngen=ngen, verbose=False)
    pareto = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
    pareto_points = [ind.fitness.values for ind in pareto]
    return pareto, pareto_points

# ----------------------------
# 9. METRICS UTILITIES
# ----------------------------
def hypervolume(points):
    if len(points) == 0:
        return 0.0
    ref = np.max(points, axis=0) + 1.0
    hv = 0.0
    for p in points:
        dx = max(ref[0] - p[0], 0)
        dy = max(ref[1] - p[1], 0)
        hv += dx * dy
    return hv

def spread(points):
    if len(points) < 2:
        return 0.0
    pts = sorted(points, key=lambda x: x[0])
    dists = [np.linalg.norm(np.array(pts[i]) - np.array(pts[i-1])) for i in range(1, len(pts))]
    return float(np.std(dists))

# ----------------------------
# 10. SCENARIO SAMPLING & RUN
# ----------------------------
# Définition des scénarios avec tes valeurs
scenario_sizes = {
    "Small": {"incidents": 50, "ambulances": 5},      # 50 incidents, 5 ambulances
    "Medium": {"incidents": 100, "ambulances": 10},    # 100 incidents, 10 ambulances
    "Large": {"incidents": 1000, "ambulances": 20}     # 1000 incidents, 20 ambulances
}

results = []

for scenario_name, params in scenario_sizes.items():
    print(f"\n=== Scenario: {scenario_name} ===")
    
    # Échantillonnage des incidents
    incidents_sample = sample_for_mapping.sample(
        min(params["incidents"], len(sample_for_mapping)), random_state=42
    )
    incidents_nodes = incidents_sample["node"].tolist()
    
    # Priorités par incident
    priorities_map = {
        row["node"]: int(row["Priority"]) if "Priority" in row else 1
        for _, row in incidents_sample.iterrows()
    }
    
    # Sélection des ambulances
    ambulance_nodes = random.sample(list(G.nodes), params["ambulances"])
    
    # NSGA-II
    t0 = time.time()
    pareto, pareto_points = run_nsga2(
        incidents_nodes, ambulance_nodes, priorities_map, G,
        pop_size=40, ngen=20, cxpb=0.7, mutpb=0.2
    )
    t1 = time.time()
    
    # RL
    env = EMSEnv(G, ambulance_nodes, demand_dict, node_counts)
    model, rewards_hist, losses_hist = train_rl(env, episodes=40, max_steps=40)
    
    # Calcul des métriques
    pareto_arr = np.array(pareto_points) if len(pareto_points) > 0 else np.zeros((1,2))
    avg_response_time = float(np.mean(pareto_arr[:, 0]))
    coverage_threshold = 8.0
    coverage_rate = float(np.sum(pareto_arr[:, 0] <= coverage_threshold) / max(1, len(pareto_arr)))
    hv = hypervolume(pareto_arr)
    div = spread(pareto_points)
    
    results.append({
        "Scenario": scenario_name,
        "Incidents": params["incidents"],
        "Ambulances": params["ambulances"],
        "Hypervolume": hv,
        "Diversity": div,
        "Avg_Response_Time": avg_response_time,
        "Coverage_Rate": coverage_rate,
        "Avg_Reward": float(np.mean(rewards_hist)),
        "Policy_Stability": float(np.std(rewards_hist)),
        "Avg_RL_Loss": float(np.mean(losses_hist)) if len(losses_hist) > 0 else 0.0,
        "Computation_Time_s": t1 - t0
    })

# Résultats
df_results = pd.DataFrame(results)
print("\n=== Résultats synthèse ===")
print(df_results)

In [ ]:
# ----------------------------
# 11. PARETO EVALUATION & FIGURES
# ----------------------------
# Plot Pareto front for last scenario (if exists)
if len(pareto_points) > 0:
    plt.figure(figsize=(6,5))
    xs = [p[0] for p in pareto_points]
    ys = [p[1] for p in pareto_points]
    plt.scatter(xs, ys, c="tab:blue", s=40)
    plt.xlabel("Total Response Time (min)")
    plt.ylabel("Priority-weighted Response Time")
    plt.title(f"Pareto Front - {scenario_name}")
    plt.grid(True)
    plt.show()

# ----------------------------
# 12. PERFORMANCE DASHBOARD FIGURES
# ----------------------------
# EMS Operational Metrics: distribution of response times (use pareto_arr)
plt.figure(figsize=(6,4))
plt.hist(pareto_arr[:,0], bins=20, color="skyblue", edgecolor="k")
plt.title("Distribution des temps de réponse (Pareto solutions)")
plt.xlabel("Temps (minutes)")
plt.ylabel("Nombre de solutions")
plt.show()

# Coverage map (scatter of node counts sample)
plt.figure(figsize=(8,6))
sample_plot = sample_for_mapping.sample(min(5000, len(sample_for_mapping)), random_state=42)
sns.scatterplot(x=sample_plot["Longitude"], y=sample_plot["Latitude"],
                size=sample_plot.groupby(["Longitude","Latitude"]).cumcount()+1,
                hue=sample_plot["Priority"], palette="viridis", alpha=0.6, legend=False)
plt.title("Extrait des incidents (taille ~5000) - couleur = priorité")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.show()

# RL Metrics: rewards and loss curves
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(rewards_hist, marker="o")
plt.title("RL: Récompenses par épisode")
plt.xlabel("Épisode"); plt.ylabel("Récompense cumulée")
plt.subplot(1,2,2)
plt.plot(losses_hist)
plt.title("RL: Loss pendant l'entraînement")
plt.xlabel("Step"); plt.ylabel("Loss")
plt.tight_layout()
plt.show()

# NSGA-II Metrics: Hypervolume bar
plt.figure(figsize=(6,4))
sns.barplot(x="Scenario", y="Hypervolume", data=df_results, palette="Greens_d")
plt.title("NSGA-II: Hypervolume par scénario")
plt.show()

# Diversity (Spread)
plt.figure(figsize=(6,4))
sns.barplot(x="Scenario", y="Diversity", data=df_results, palette="mako")
plt.title("NSGA-II: Diversity (Spread)")
plt.show()

# Spatio-temporal Metrics: Hotspot comparison heatmap (pred vs true)
# Build small dataframe for plotting top nodes
top_nodes_df = node_counts.sort_values("count", ascending=False).head(200)
top_nodes_df = top_nodes_df.reset_index(drop=True)
# get coordinates for nodes
coords = [node_xy.get(n, (np.nan, np.nan)) for n in top_nodes_df["node"].values]
top_nodes_df["x"] = [c[0] for c in coords]
top_nodes_df["y"] = [c[1] for c in coords]

plt.figure(figsize=(8,6))
plt.scatter(top_nodes_df["x"], top_nodes_df["y"], c=top_nodes_df["count"], cmap="Reds", s=40)
plt.colorbar(label="Incident count")
plt.title("Top nodes (counts) - hotspots réels (approx.)")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.show()

# MAE / RMSE bar
plt.figure(figsize=(6,4))
melt = df_results.melt(id_vars=["Scenario"], value_vars=["MAE", "RMSE"], var_name="Metric", value_name="Value")
sns.barplot(x="Scenario", y="Value", hue="Metric", data=melt)
plt.title("Spatio-temporal errors (MAE / RMSE)")
plt.show()

# ----------------------------
# 13. Export / Résumé
# ----------------------------
print("\nFinal metrics table:")
print(df_results.to_string(index=False))